Esercizio: implementa un ciclo di training in Keras utilizzando contemporaneamente ModelCheckpoint (per salvare 'best_model.kera') e ReduceLROnPlateau. Crea inoltre una Custom Callback chiamata 'SimpleLogger' che stampi il valore del Learning Rate alla fine di ogni epoca. Suggerimento: per accedere al LR nella callback usa 'float(tf.keras.backend.get_valut(self.model.optimizer.lr))'

In [1]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# =================================================================
# 1. PREPARAZIONE DEL DATASET
# =================================================================
X, y = make_classification(n_samples=2000, n_features=30, n_informative=20, 
                           n_redundant=10, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# =================================================================
# 2. CUSTOM CALLBACK: LOG LR E MONITORAGGIO PESI
# =================================================================
class SimplesLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        # 1. Recupero Learning Rate
        lr = float(tf.keras.backend.get_value(self.model.optimizer.learning_rate))
        
        # 2. Monitoraggio Pesi (Prendiamo il primo strato Dense dopo l'Input)
        # Calcoliamo la media dei valori assoluti dei pesi per vedere quanto sono "grandi"
        weights, biases = self.model.layers[0].get_weights()
        avg_weight = np.mean(np.abs(weights))
        
        print(f"\n - [INFO] Fine Epoca {epoch+1}:")
        print(f"   > Learning Rate: {lr:.6f}")
        print(f"   > Media abs pesi (Layer 1): {avg_weight:.6f}")

# =================================================================
# 3. CONFIGURAZIONE CALLBACK
# =================================================================

# Salva il file del modello solo quando migliora la val_loss (min), andando a sovrascrivere il precedente
checkpoint_cb = ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# Riduce il LR se la loss stalla per 3 epoche, in modo che il modello si possa adattare meglio quando si avvicina al valore minimo
reduce_lr_cb = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,  # Riduce il LR del 20% quando la val_loss non migliora
    patience=3, #il modello non migliora per 3 epoche, riduci il LR
    min_lr=1e-6, #valore minimo del LR per evitare di scendere troppo
    verbose=1
)

# STOPPA il modello se non migliora per 10 epoche
early_stopping_cb = EarlyStopping(
    monitor='val_loss',
    patience=10,        # Numero di epoche da aspettare
    mode='min',
    restore_best_weights=True, # Al termine, ripristina i pesi migliori invece degli ultimi
    verbose=1
)

# =================================================================
# 4. COSTRUZIONE E TRAINING
# =================================================================
model = Sequential([
    Input(shape=(30,)),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid') #dens 1 perchè è un problema di classificazione binaria, sigmoid per avere output tra 0 e 1
    ])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\nInizio addestramento con Early Stopping e Monitoraggio Pesi...")

history = model.fit(
    X_train, y_train,
    epochs=100, # Aumentiamo le epoche potenziali, tanto lo stop è automatico
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[checkpoint_cb, reduce_lr_cb, early_stopping_cb, SimplesLogger()],
    verbose=0 
)


Inizio addestramento con Early Stopping e Monitoraggio Pesi...

Epoch 1: val_loss improved from None to 0.42454, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras

 - [INFO] Fine Epoca 1:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.095832

Epoch 2: val_loss improved from 0.42454 to 0.32573, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras

 - [INFO] Fine Epoca 2:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.096007

Epoch 3: val_loss improved from 0.32573 to 0.26201, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras

 - [INFO] Fine Epoca 3:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.096353

Epoch 4: val_loss improved from 0.26201 to 0.22397, saving model to best_model.keras

Epoch 4: finished saving model to best_model.keras

 - [INFO] Fine Epoca 4:
   > Learning Rate: 0.001000
   > Media abs pesi (Layer 1): 0.096589

Epoch 5: v